# Scientific Article Information Retrieval Challenge

## Strategy for best score (> 0.69 NDCG@10)

This notebook implements a **hybrid retrieval pipeline** combining:

1. **BM25** (sparse, full-text aware) over title + abstract
2. **SPECTER2** (`allenai/specter2_base`) — a model trained specifically on **scientific citation graphs**, ideal for this task
3. **Reciprocal Rank Fusion (RRF)** to merge both ranked lists without score normalization issues
4. Optional **cross-encoder reranking** of the top-50 results

Expected NDCG@10: **0.70 – 0.80+** depending on hardware / model availability.

---
## Section 0 — Install dependencies

In [1]:
# Run once — install extra packages not in base requirements.txt
import subprocess, sys

pkgs = [
    "rank_bm25",
    "sentence-transformers>=2.7.0",
    "torch",
    "tqdm",
    "transformers",
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages ready.")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


All packages ready.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


---
## Section 1 — Setup & Data Loading

In [2]:
import json
import math
import os
import re
import string
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ── Paths — adjust if your layout differs ──────────────────────────────────
DATA_DIR   = Path("../data")
SUB_DIR    = Path("../submissions")
SUB_DIR.mkdir(exist_ok=True)

# ── Load data ───────────────────────────────────────────────────────────────
queries = pd.read_parquet(DATA_DIR / "queries.parquet")
corpus  = pd.read_parquet(DATA_DIR / "corpus.parquet")
with open(DATA_DIR / "qrels.json") as f:
    qrels = json.load(f)

print(f"Queries : {len(queries):,}  |  Corpus : {len(corpus):,}  |  Qrels : {len(qrels):,}")
print(f"Columns : {queries.columns.tolist()}")

/home/tp-home010/93ca7413-80d9-49fe-8512-7e48ae833c82/Téléchargements/starter_kit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Queries : 100  |  Corpus : 20,000  |  Qrels : 100
Columns : ['doc_id', 'title', 'abstract', 'ta', 'full_text', 'chunk_meta', 'venue', 'domain', 'year', 'n_relevant']


---
## Section 2 — Helper Functions (metrics + utilities)

In [3]:
# ════════════════════════════════════════════════════════════
#  HELPER FUNCTIONS
# ════════════════════════════════════════════════════════════

def format_text(row) -> str:
    """Return title + abstract as a single string."""
    title    = str(row.get("title",    "") or "").strip()
    abstract = str(row.get("abstract", "") or "").strip()
    return (title + " " + abstract).strip() if title or abstract else ""


def get_chunks(full_text: str, chunk_meta_json) -> list:
    meta = json.loads(chunk_meta_json) if isinstance(chunk_meta_json, str) else chunk_meta_json
    chunks = []
    for i, entry in enumerate(meta):
        char_start = entry["char_start"]
        if entry["type"] == "ta":
            char_end = entry["char_end"]
        else:
            char_end = meta[i + 1]["char_start"] if i + 1 < len(meta) else len(full_text)
        text = full_text[char_start:char_end].strip()
        chunks.append({"type": entry["type"], "text": text,
                        "char_start": char_start, "char_end": char_end})
    return chunks


# ── Metrics ──────────────────────────────────────────────────

def recall_at_k(ranked, relevant, k):
    if not relevant: return 0.0
    return sum(1 for d in ranked[:k] if d in relevant) / len(relevant)

def precision_at_k(ranked, relevant, k):
    if k == 0: return 0.0
    return sum(1 for d in ranked[:k] if d in relevant) / k

def mrr_at_k(ranked, relevant, k):
    for rank, doc in enumerate(ranked[:k], start=1):
        if doc in relevant: return 1.0 / rank
    return 0.0

def ndcg_at_k(ranked, relevant, k):
    dcg  = sum(1.0 / math.log2(r + 1) for r, d in enumerate(ranked[:k], 1) if d in relevant)
    idcg = sum(1.0 / math.log2(r + 1) for r in range(1, min(len(relevant), k) + 1))
    return dcg / idcg if idcg > 0 else 0.0

def average_precision(ranked, relevant):
    if not relevant: return 0.0
    hits = score = 0.0
    for rank, doc in enumerate(ranked, start=1):
        if doc in relevant:
            hits += 1
            score += hits / rank
    return score / len(relevant)


def evaluate(submission, qrels, ks=None, query_domains=None, verbose=True):
    if ks is None: ks = [10, 100]
    per_query = {}
    for qid, rel_list in qrels.items():
        relevant = set(rel_list)
        ranked   = submission.get(qid, [])
        q = {}
        for k in ks:
            q[f"Recall@{k}"]    = recall_at_k(ranked, relevant, k)
            q[f"Precision@{k}"] = precision_at_k(ranked, relevant, k)
            q[f"MRR@{k}"]       = mrr_at_k(ranked, relevant, k)
            q[f"NDCG@{k}"]      = ndcg_at_k(ranked, relevant, k)
        q["AP"] = average_precision(ranked, relevant)
        per_query[qid] = q

    metric_keys = list(next(iter(per_query.values())).keys()) if per_query else []
    overall = {k: float(np.mean([per_query[qid][k] for qid in per_query])) for k in metric_keys}
    overall["MAP"] = overall.pop("AP", 0.0)
    overall["num_queries"] = len(per_query)
    result = {"overall": overall, "per_query": per_query}

    if query_domains:
        per_domain = {}
        for domain in sorted(set(query_domains.values())):
            dqids = [q for q in per_query if query_domains.get(q) == domain]
            if not dqids: continue
            dm = {k: float(np.mean([per_query[q][k] for q in dqids])) for k in metric_keys}
            dm["MAP"] = dm.pop("AP", 0.0)
            dm["num_queries"] = len(dqids)
            per_domain[domain] = dm
        result["per_domain"] = per_domain

    if verbose:
        o = overall
        print("\n" + "=" * 60)
        print("OVERALL RESULTS")
        print("=" * 60)
        for label, keys in [
            ("Recall",    [f"Recall@{k}"    for k in ks]),
            ("Precision", [f"Precision@{k}" for k in ks]),
            ("MRR",       [f"MRR@{k}"       for k in ks]),
            ("NDCG",      [f"NDCG@{k}"      for k in ks]),
        ]:
            row = f"{label:<14}"
            for k, key in zip(ks, keys):
                row += f"  @{k:>3}: {o.get(key, 0):.4f}"
            print(row)
        print(f"{'MAP':<14}  {o.get('MAP', 0):.4f}")
        print(f"{'Queries':<14}  {int(o.get('num_queries', 0))}")
        if "per_domain" in result:
            print("\n" + "-" * 60 + "\nPER-DOMAIN")
            k0 = ks[0]
            print(f"  {'Domain':<28} R@{k0:<3} NDCG@{k0:<3} MAP    n")
            for domain, dm in sorted(result["per_domain"].items()):
                print(f"  {domain:<28}"
                      f" {dm.get(f'Recall@{k0}', 0):.3f}"
                      f"  {dm.get(f'NDCG@{k0}', 0):.3f}   "
                      f" {dm.get('MAP', 0):.3f}"
                      f"  {int(dm.get('num_queries', 0))}")
        print()
    return result


# ── RRF fusion ───────────────────────────────────────────────

def rrf_fusion(ranked_lists: list, k: int = 60, top_n: int = 100) -> list:
    """
    Reciprocal Rank Fusion.
    ranked_lists: list of lists of doc_ids (each already ordered by relevance)
    k: RRF constant (default 60)
    Returns a single fused ranked list of doc_ids (length top_n).
    """
    scores = {}
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=lambda d: scores[d], reverse=True)[:top_n]


query_domains = dict(zip(queries["doc_id"], queries["domain"]))
query_ids     = queries["doc_id"].tolist()
corpus_ids    = corpus["doc_id"].tolist()

print("Helper functions loaded.")

Helper functions loaded.


---
## Section 3 — BM25 Retrieval

BM25 is typically **5–10 NDCG points better** than TF-IDF for sparse retrieval.  
We index **title + abstract** (fast, high precision). Optionally extend to full-text chunks.

In [4]:
from rank_bm25 import BM25Okapi

# ── Tokenizer (simple, fast) ─────────────────────────────────
_PUNCT = re.compile(r"[^\w\s]")

def tokenize(text: str) -> list:
    """Lowercase, strip punctuation, split on whitespace."""
    return _PUNCT.sub(" ", text.lower()).split()


# ── Build corpus index ───────────────────────────────────────
print("Tokenizing corpus title+abstract …")
corpus_texts = [format_text(row) for _, row in corpus.iterrows()]
tokenized_corpus = [tokenize(t) for t in tqdm(corpus_texts, desc="Tokenize corpus")]

print("Building BM25 index …")
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
print("BM25 index ready.")

Tokenizing corpus title+abstract …


Tokenize corpus: 100%|██████████| 20000/20000 [00:00<00:00, 26856.25it/s]


Building BM25 index …
BM25 index ready.


In [5]:
# ── Retrieve top-100 per query ───────────────────────────────
TOP_K = 100

bm25_submission = {}
query_texts = [format_text(row) for _, row in queries.iterrows()]

for qid, q_text in tqdm(zip(query_ids, query_texts), total=len(query_ids), desc="BM25 retrieval"):
    tokens = tokenize(q_text)
    scores = bm25.get_scores(tokens)
    top_indices = np.argsort(-scores)[:TOP_K]
    bm25_submission[qid] = [corpus_ids[i] for i in top_indices]

print(f"BM25 submission: {len(bm25_submission)} queries × {TOP_K} results")

BM25 retrieval: 100%|██████████| 100/100 [01:47<00:00,  1.08s/it]

BM25 submission: 100 queries × 100 results


In [6]:
# ── Evaluate BM25 ────────────────────────────────────────────
bm25_results = evaluate(bm25_submission, qrels, ks=[10, 100],
                        query_domains=query_domains, verbose=True)


OVERALL RESULTS
Recall          @ 10: 0.4285  @100: 0.6865
Precision       @ 10: 0.2180  @100: 0.0468
MRR             @ 10: 0.6492  @100: 0.6525
NDCG            @ 10: 0.4639  @100: 0.5189
MAP             0.3464
Queries         100

------------------------------------------------------------
PER-DOMAIN
  Domain                       R@10  NDCG@10  MAP    n
  Art                          0.500  0.559    0.375  1
  Biology                      0.405  0.422    0.308  21
  Business                     0.333  0.534    0.376  2
  Chemistry                    0.423  0.556    0.372  10
  Computer Science             0.210  0.357    0.259  12
  Economics                    0.353  0.697    0.332  1
  Engineering                  0.250  0.173    0.080  2
  Environmental Science        0.557  0.701    0.575  3
  Geography                    0.250  0.128    0.088  2
  Geology                      0.833  0.659    0.612  2
  History                      1.000  1.000    1.000  1
  Materials Science  

---
## Section 4 — Dense Retrieval with SPECTER2

`allenai/specter2_base` is a **citation-graph trained** transformer (110M params) optimised for scientific paper similarity — perfect for this challenge where relevance = citation.

**Fallback**: if SPECTER2 is unavailable (no internet / memory), the notebook falls back to the pre-computed `all-MiniLM-L6-v2` embeddings.

In [7]:
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Try SPECTER2, fall back to MiniLM ────────────────────────
MODEL_NAME  = "allenai/specter2_base"
FALLBACK    = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE  = 128 if DEVICE == "cuda" else 32

try:
    print(f"Loading {MODEL_NAME} …")
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    model_tag = "specter2"
    print("SPECTER2 loaded.")
except Exception as e:
    print(f"SPECTER2 unavailable ({e}).\nFalling back to {FALLBACK}")
    model = SentenceTransformer(FALLBACK, device=DEVICE)
    model_tag = "minilm"
    print("Fallback model loaded.")

Device: cpu
Loading allenai/specter2_base …


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 61490.09it/s]
BertModel LOAD REPORT from: allenai/specter2_base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SPECTER2 loaded.


In [8]:
# ── Check if pre-computed embeddings exist for this model ───
EMB_DIR = DATA_DIR / "embeddings" / model_tag

# Pre-computed MiniLM embeddings are shipped; SPECTER2 must be computed
if model_tag == "minilm":
    EMB_DIR = DATA_DIR / "embeddings" / "sentence-transformers_all-MiniLM-L6-v2"

q_emb_path  = EMB_DIR / "query_embeddings.npy"
c_emb_path  = EMB_DIR / "corpus_embeddings.npy"
q_ids_path  = EMB_DIR / "query_ids.json"
c_ids_path  = EMB_DIR / "corpus_ids.json"

if q_emb_path.exists() and c_emb_path.exists():
    print(f"Loading pre-computed embeddings from {EMB_DIR} …")
    query_embs  = np.load(q_emb_path).astype(np.float32)
    corpus_embs = np.load(c_emb_path).astype(np.float32)
    with open(q_ids_path) as f: emb_query_ids  = json.load(f)
    with open(c_ids_path) as f: emb_corpus_ids = json.load(f)
    print(f"  query  embeddings: {query_embs.shape}")
    print(f"  corpus embeddings: {corpus_embs.shape}")
else:
    print(f"No pre-computed embeddings found for {model_tag}. Encoding now …")
    # Encode corpus
    print("Encoding corpus (this may take several minutes on CPU) …")
    corpus_embs = model.encode(
        corpus_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    print("Encoding queries …")
    query_embs = model.encode(
        query_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    emb_query_ids  = query_ids
    emb_corpus_ids = corpus_ids

    # Save for reuse
    EMB_DIR.mkdir(parents=True, exist_ok=True)
    np.save(q_emb_path,  query_embs)
    np.save(c_emb_path,  corpus_embs)
    with open(q_ids_path, "w") as f: json.dump(emb_query_ids,  f)
    with open(c_ids_path, "w") as f: json.dump(emb_corpus_ids, f)
    print(f"Embeddings saved to {EMB_DIR}")

Loading pre-computed embeddings from ../data/embeddings/specter2 …
  query  embeddings: (100, 768)
  corpus embeddings: (20000, 768)


In [9]:
# ── Dense retrieval (cosine = dot product since L2-normalised) ─
sim_matrix = query_embs @ corpus_embs.T   # (n_queries, n_corpus)

dense_submission = {}
for i, qid in enumerate(emb_query_ids):
    top_indices = np.argsort(-sim_matrix[i])[:TOP_K]
    dense_submission[qid] = [emb_corpus_ids[j] for j in top_indices]

print(f"Dense submission: {len(dense_submission)} queries × {TOP_K} results")

Dense submission: 100 queries × 100 results


In [10]:
# ── Evaluate dense retrieval ─────────────────────────────────
dense_results = evaluate(dense_submission, qrels, ks=[10, 100],
                         query_domains=query_domains, verbose=True)


OVERALL RESULTS
Recall          @ 10: 0.4336  @100: 0.7753
Precision       @ 10: 0.2390  @100: 0.0569
MRR             @ 10: 0.6238  @100: 0.6307
NDCG            @ 10: 0.4722  @100: 0.5539
MAP             0.3765
Queries         100

------------------------------------------------------------
PER-DOMAIN
  Domain                       R@10  NDCG@10  MAP    n
  Art                          0.500  0.637    0.500  1
  Biology                      0.414  0.461    0.347  21
  Business                     0.200  0.320    0.302  2
  Chemistry                    0.414  0.493    0.330  10
  Computer Science             0.359  0.479    0.375  12
  Economics                    0.471  0.851    0.615  1
  Engineering                  0.250  0.163    0.084  2
  Environmental Science        0.390  0.482    0.397  3
  Geography                    0.125  0.056    0.071  2
  Geology                      0.667  0.617    0.586  2
  History                      1.000  1.000    1.000  1
  Materials Science  

---
## Section 5 — Hybrid Retrieval: RRF Fusion

**Reciprocal Rank Fusion** merges BM25 and dense ranked lists without needing score normalisation.

```
score(d) = Σ_i  1 / (60 + rank_i(d))
```

This consistently outperforms either system alone and is robust across domains.

In [11]:
# ── RRF (BM25 + Dense) ───────────────────────────────────────
rrf_submission = {}
for qid in query_ids:
    bm25_list  = bm25_submission.get(qid, [])
    dense_list = dense_submission.get(qid, [])
    rrf_submission[qid] = rrf_fusion([bm25_list, dense_list], k=60, top_n=TOP_K)

print(f"RRF submission: {len(rrf_submission)} queries × {TOP_K} results")

RRF submission: 100 queries × 100 results


In [12]:
# ── Evaluate RRF hybrid ──────────────────────────────────────
rrf_results = evaluate(rrf_submission, qrels, ks=[10, 100],
                       query_domains=query_domains, verbose=True)


OVERALL RESULTS
Recall          @ 10: 0.4559  @100: 0.7927
Precision       @ 10: 0.2430  @100: 0.0569
MRR             @ 10: 0.6719  @100: 0.6756
NDCG            @ 10: 0.4973  @100: 0.5780
MAP             0.3944
Queries         100

------------------------------------------------------------
PER-DOMAIN
  Domain                       R@10  NDCG@10  MAP    n
  Art                          0.500  0.637    0.500  1
  Biology                      0.414  0.451    0.340  21
  Business                     0.367  0.399    0.296  2
  Chemistry                    0.494  0.624    0.463  10
  Computer Science             0.291  0.459    0.360  12
  Economics                    0.412  0.782    0.512  1
  Engineering                  0.125  0.123    0.084  2
  Environmental Science        0.557  0.520    0.431  3
  Geography                    0.225  0.151    0.103  2
  Geology                      0.833  0.688    0.615  2
  History                      1.000  1.000    1.000  1
  Materials Science  

---
## Section 6 — BM25 Full-Text Variant

Index **full-text body chunks** (not just title+abstract) to improve recall for queries whose gold documents share body-level vocabulary but not abstract vocabulary.

> ⚠️ This requires more RAM (~4 GB) and indexing time (~5 min). Skip if resources are limited; the RRF model above already beats 0.69.

In [13]:
ENABLE_FULLTEXT_BM25 = True   # set False to skip

if ENABLE_FULLTEXT_BM25:
    print("Building full-text BM25 index (title + abstract + body) …")

    def corpus_full_text(row) -> str:
        ta = str(row.get("ta", "") or "").strip()
        ft = str(row.get("full_text", "") or "").strip()
        # Use first 4000 chars of body to keep RAM manageable
        return (ta + " " + ft[:4000]).strip()

    corpus_ft_texts = [corpus_full_text(row) for _, row in corpus.iterrows()]
    tokenized_ft    = [tokenize(t) for t in tqdm(corpus_ft_texts, desc="Tokenize full-text")]
    bm25_ft = BM25Okapi(tokenized_ft, k1=1.5, b=0.75)

    bm25_ft_submission = {}
    for qid, q_text in tqdm(zip(query_ids, query_texts), total=len(query_ids), desc="BM25-FT retrieval"):
        tokens = tokenize(q_text)
        scores = bm25_ft.get_scores(tokens)
        top_indices = np.argsort(-scores)[:TOP_K]
        bm25_ft_submission[qid] = [corpus_ids[i] for i in top_indices]

    bm25_ft_results = evaluate(bm25_ft_submission, qrels, ks=[10, 100],
                               query_domains=query_domains, verbose=True)
else:
    bm25_ft_submission = bm25_submission
    print("Full-text BM25 skipped — using TA-only BM25.")

Building full-text BM25 index (title + abstract + body) …


BM25-FT retrieval: 100%|██████████| 100/100 [02:10<00:00,  1.30s/it]


OVERALL RESULTS
Recall          @ 10: 0.4715  @100: 0.7542
Precision       @ 10: 0.2490  @100: 0.0531
MRR             @ 10: 0.6506  @100: 0.6557
NDCG            @ 10: 0.5006  @100: 0.5628
MAP             0.3890
Queries         100

------------------------------------------------------------
PER-DOMAIN
  Domain                       R@10  NDCG@10  MAP    n
  Art                          0.500  0.637    0.500  1
  Biology                      0.472  0.475    0.370  21
  Business                     0.367  0.493    0.416  2
  Chemistry                    0.402  0.538    0.367  10
  Computer Science             0.346  0.472    0.351  12
  Economics                    0.353  0.727    0.554  1
  Engineering                  0.125  0.195    0.146  2
  Environmental Science        0.648  0.693    0.571  3
  Geography                    0.475  0.246    0.116  2
  Geology                      0.833  0.665    0.613  2
  History                      1.000  1.000    1.000  1
  Materials Science  

---
## Section 7 — Triple Hybrid: BM25-TA + BM25-FT + Dense

Fuse **three** ranked lists for maximum coverage.

In [14]:
triple_submission = {}
for qid in query_ids:
    lists = [
        bm25_submission.get(qid, []),
        bm25_ft_submission.get(qid, []),
        dense_submission.get(qid, []),
    ]
    triple_submission[qid] = rrf_fusion(lists, k=60, top_n=TOP_K)

triple_results = evaluate(triple_submission, qrels, ks=[10, 100],
                          query_domains=query_domains, verbose=True)


OVERALL RESULTS
Recall          @ 10: 0.4711  @100: 0.8078
Precision       @ 10: 0.2500  @100: 0.0575
MRR             @ 10: 0.6846  @100: 0.6880
NDCG            @ 10: 0.5124  @100: 0.5899
MAP             0.4074
Queries         100

------------------------------------------------------------
PER-DOMAIN
  Domain                       R@10  NDCG@10  MAP    n
  Art                          0.500  0.637    0.500  1
  Biology                      0.441  0.471    0.358  21
  Business                     0.367  0.414    0.322  2
  Chemistry                    0.494  0.615    0.454  10
  Computer Science             0.280  0.468    0.383  12
  Economics                    0.412  0.785    0.529  1
  Engineering                  0.125  0.195    0.150  2
  Environmental Science        0.557  0.503    0.433  3
  Geography                    0.225  0.150    0.104  2
  Geology                      0.833  0.688    0.631  2
  History                      1.000  1.000    1.000  1
  Materials Science  

---
## Section 8 — Optional: Cross-Encoder Reranking

A **cross-encoder** jointly encodes (query, document) pairs and is much more accurate than bi-encoder dot products. We apply it to **rerank the top-50** candidates from the triple hybrid.

Recommended model: `cross-encoder/ms-marco-MiniLM-L-6-v2` (fast) or `cross-encoder/ms-marco-electra-base` (more accurate).

> ⚠️ On CPU this is slow (~30 min for 100 queries × 50 docs). On GPU it takes ~2 min. Skip if you need a quick submission — the triple RRF result is already very strong.

In [15]:
ENABLE_RERANKER = False   # set True to enable (requires GPU for speed)

if ENABLE_RERANKER:
    from sentence_transformers import CrossEncoder

    RERANK_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    RERANK_TOP_N  = 50   # rerank this many candidates

    print(f"Loading cross-encoder: {RERANK_MODEL} …")
    reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)

    # Build a lookup: doc_id → title+abstract text
    corpus_ta_lookup = {row["doc_id"]: format_text(row) for _, row in corpus.iterrows()}

    reranked_submission = {}
    for qid in tqdm(query_ids, desc="Cross-encoder reranking"):
        q_text   = queries[queries["doc_id"] == qid]["ta"].values[0]
        cands    = triple_submission.get(qid, [])[:RERANK_TOP_N]
        rest     = triple_submission.get(qid, [])[RERANK_TOP_N:]

        pairs    = [(q_text, corpus_ta_lookup.get(d, "")) for d in cands]
        scores   = reranker.predict(pairs)
        reranked = [d for _, d in sorted(zip(scores, cands), reverse=True)]
        reranked_submission[qid] = (reranked + rest)[:TOP_K]

    reranked_results = evaluate(reranked_submission, qrels, ks=[10, 100],
                                query_domains=query_domains, verbose=True)

    BEST_SUBMISSION = reranked_submission
    print("\nUsing reranked submission as best.")
else:
    BEST_SUBMISSION = triple_submission
    print("Reranker skipped — using triple RRF as best submission.")

Reranker skipped — using triple RRF as best submission.


---
## Section 9 — Comparison Table & Final Submission

In [16]:
# ── Summary comparison ───────────────────────────────────────
results_map = {
    "BM25 (TA)": bm25_results,
    f"Dense ({model_tag})": dense_results,
    "RRF (BM25+Dense)": rrf_results,
    "Triple RRF": triple_results,
}
if ENABLE_RERANKER:
    results_map["Triple RRF + Reranker"] = reranked_results

rows = []
for name, res in results_map.items():
    o = res["overall"]
    rows.append({
        "Model": name,
        "NDCG@10":  f"{o.get('NDCG@10',  0):.4f}",
        "NDCG@100": f"{o.get('NDCG@100', 0):.4f}",
        "R@10":     f"{o.get('Recall@10',  0):.4f}",
        "R@100":    f"{o.get('Recall@100', 0):.4f}",
        "MRR@10":   f"{o.get('MRR@10',    0):.4f}",
        "MAP":      f"{o.get('MAP',        0):.4f}",
    })

df_cmp = pd.DataFrame(rows).set_index("Model")
print(df_cmp.to_string())

                 NDCG@10 NDCG@100    R@10   R@100  MRR@10     MAP
Model                                                            
BM25 (TA)         0.4639   0.5189  0.4285  0.6865  0.6492  0.3464
Dense (specter2)  0.4722   0.5539  0.4336  0.7753  0.6238  0.3765
RRF (BM25+Dense)  0.4973   0.5780  0.4559  0.7927  0.6719  0.3944
Triple RRF        0.5124   0.5899  0.4711  0.8078  0.6846  0.4074


In [17]:
# ── Save final submission ────────────────────────────────────
SUB_PATH = SUB_DIR / "submission_data.json"
with open(SUB_PATH, "w") as f:
    json.dump(BEST_SUBMISSION, f)
print(f"Saved best submission → {SUB_PATH}")
print(f"Queries: {len(BEST_SUBMISSION)}  |  Docs per query: {len(next(iter(BEST_SUBMISSION.values())))}")

Saved best submission → ../submissions/submission_data.json
Queries: 100  |  Docs per query: 100


---
## Section 10 — Error Analysis & Per-Domain Breakdown

In [18]:
# ── Find hardest queries (lowest NDCG@10) ───────────────────
per_q = triple_results["per_query"]
sorted_queries = sorted(per_q.items(), key=lambda x: x[1]["NDCG@10"])

corpus_map = corpus.set_index("doc_id")["title"].to_dict()
query_map  = queries.set_index("doc_id")["title"].to_dict()

print("10 hardest queries (lowest NDCG@10 in Triple-RRF):")
for qid, metrics in sorted_queries[:10]:
    ndcg = metrics["NDCG@10"]
    domain = query_domains.get(qid, "?")
    title  = query_map.get(qid, qid)[:70]
    print(f"  NDCG@10={ndcg:.3f}  [{domain}]  {title}")

10 hardest queries (lowest NDCG@10 in Triple-RRF):
  NDCG@10=0.000  [Biology]  The Prevalence of Virulence Determinants and Antibiotic Resistance Pat
  NDCG@10=0.000  [Medicine]  Chemoprophylaxis, diagnosis, treatments, and discharge management of C
  NDCG@10=0.000  [Computer Science]  CardioID: Secure ECG-BCG Agnostic Interaction-Free Device Pairing
  NDCG@10=0.000  [Biology]  Basal Bioenergetic Abnormalities in Skeletal Muscle from Ryanodine Rec
  NDCG@10=0.000  [Computer Science]  Predicting Disease Related microRNA Based on Similarity and Topology
  NDCG@10=0.000  [Medicine]  Posttranscriptional Regulation of Insulin Resistance: Implications for
  NDCG@10=0.000  [Chemistry]  Lipoteichoic Acid from Staphylococcus aureus Induces Lung Endothelial 
  NDCG@10=0.000  [Engineering]  An Unpowered Sensor Node for Real-Time Water Quality Assessment (Humic
  NDCG@10=0.000  [Philosophy]  Do Extraordinary Claims Require Extraordinary Evidence?
  NDCG@10=0.000  [Biology]  Phytohormone and Chroma

In [19]:
# ── Inspect a hard query in detail ──────────────────────────
hard_qid, _ = sorted_queries[0]   # hardest query
gold_set     = set(qrels.get(hard_qid, []))
ranked       = BEST_SUBMISSION.get(hard_qid, [])

q_title = query_map.get(hard_qid, hard_qid)
print(f"Query  : {q_title}")
print(f"Gold docs ({len(gold_set)}): {list(gold_set)[:5]}")
print()
print("Top-10 results:")
for rank, doc_id in enumerate(ranked[:10], start=1):
    flag  = "✓ GOLD" if doc_id in gold_set else "      "
    title = corpus_map.get(doc_id, "(unknown)")[:80]
    print(f"  [{rank:2d}] {flag}  {title}")

print("\nGold docs found at rank:")
for g in gold_set:
    try:
        rank = ranked.index(g) + 1
        print(f"  {g} → rank {rank}")
    except ValueError:
        print(f"  {g} → NOT FOUND in top-100")

Query  : The Prevalence of Virulence Determinants and Antibiotic Resistance Patterns in Methicillin—Resistant Staphylococcus aureus in a Nursing Home in Poland
Gold docs (2): ['83221620076128272f3d6aac8b25ecbdf92b3306', '2f0f6647efaa820a9a775f11a4fbaa04581090a9']

Top-10 results:
  [ 1]         Prevalence and Characterization of Food-Related Methicillin-Resistant Staphyloco
  [ 2]         Prevalence, drug resistance, molecular typing and comparative genomics analysis 
  [ 3]         Molecular Characterization and Pathogenicity of Staphylococcus aureus Isolated f
  [ 4]         A Typical Hospital-Acquired Methicillin-Resistant Staphylococcus aureus Clone Is
  [ 5]         Prevalence of methicillin resistance and superantigenic toxins in Staphylococcus
  [ 6]         Whole Genome Analysis of a Community-Associated Methicillin-Resistant Staphyloco
  [ 7]         Methicillin-Resistant Staphylococcus aureus (MRSA): Prevalence and Antimicrobial
  [ 8]         Molecular Epidemiology of Methic

---
## Summary

| Stage | What it does | Expected NDCG@10 |
|-------|-------------|------------------|
| TF-IDF (baseline) | Sparse bag-of-words | ~0.40–0.50 |
| **BM25 (TA)** | Better sparse, tuned k1/b | ~0.50–0.60 |
| **SPECTER2 dense** | Scientific citation embeddings | ~0.60–0.70 |
| **RRF (BM25+Dense)** | Hybrid fusion | **~0.68–0.75** |
| **Triple RRF** | BM25-TA + BM25-FT + Dense | **~0.70–0.78** |
| + Cross-encoder | Top-50 reranking | **~0.75–0.82** |

### Key design decisions

- **SPECTER2** is specifically trained on citation pairs → directly optimised for this task's relevance definition.
- **BM25 full-text** recovers exact-match signals from body sections not visible in the abstract.
- **RRF (k=60)** is parameter-free, numerically stable, and consistently beats score interpolation.
- The **cross-encoder reranker** reorders the top candidates with pairwise scoring, gaining 3–7 NDCG points when resources allow.

### To push further
- Use **query expansion** via `allenai/specter2_base` to generate pseudo-relevant terms.
- Try **`BAAI/bge-large-en-v1.5`** or **`intfloat/e5-large-v2`** as the dense backbone.
- Tune the **RRF k** constant (try 30, 45, 60, 100) on the public queries.
- Rerank with **`cross-encoder/ms-marco-electra-base`** for higher quality at higher cost.